In [ ]:
!nvidia-smi

In [ ]:
!pip install ultralytics albumentations scikit-learn -q
import torch
print(f'PyTorch {torch.__version__}  CUDA={torch.cuda.is_available()}')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import csv, random, time
from pathlib import Path

# ── Paths (แก้ให้ตรงกับ Drive ของคุณ) ────────────────────────────────────────
DRIVE    = '/content/drive/MyDrive/Models/Ohm-Vision'
DATA_DIR = Path(f'{DRIVE}/Classifications/cls_dataset')  # output ของ cls_dataset_prepare.py
OUT_DIR  = Path(f'{DRIVE}/Classifications/results')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Fair Comparison Constants (เหมือนกันทุกโมเดล) ────────────────────────────
SEED       = 42
MAX_EPOCHS = 100
PATIENCE   = 15
BATCH      = 32
LR         = 1e-3

# ── CSV summary helper ────────────────────────────────────────────────────────
_CSV    = OUT_DIR / 'comparison.csv'
_FIELDS = ['backbone', 'test_acc', 'best_val_acc', 'stopped_epoch', 'max_epochs',
           'params_M', 'size_mb', 'ms_per_image_cpu']

def save_summary(backbone, test_acc, best_val_acc, stopped_epoch,
                 params_m, size_mb, ms_cpu):
    write_header = not _CSV.exists()
    with open(_CSV, 'a', newline='') as f:
        w = csv.DictWriter(f, fieldnames=_FIELDS)
        if write_header:
            w.writeheader()
        w.writerow({
            'backbone':         backbone,
            'test_acc':         round(test_acc, 4),
            'best_val_acc':     round(best_val_acc, 4),
            'stopped_epoch':    stopped_epoch,
            'max_epochs':       MAX_EPOCHS,
            'params_M':         round(params_m, 2),
            'size_mb':          round(size_mb, 2),
            'ms_per_image_cpu': round(ms_cpu, 2),
        })
    print(f'[saved] {backbone}  test={test_acc:.4f}  val={best_val_acc:.4f}  cpu={ms_cpu:.1f}ms')

print(f'DATA_DIR = {DATA_DIR}')
print(f'OUT_DIR  = {OUT_DIR}')

---
## Shared utilities
Transform, Dataset wrapper, CPU inference timer — ใช้ร่วมกันทุกโมเดล

In [ ]:
import cv2
import numpy as np
import torch
import torch.nn as nn
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets
import torchvision.models as M


def _seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


_MEAN = [0.485, 0.456, 0.406]
_STD  = [0.229, 0.224, 0.225]

# normalize + tensor เท่านั้น — dataset ถูก augment offline แล้วใน cls_dataset_prepare.py
_transform = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=_MEAN, std=_STD),
    ToTensorV2(),
])


class AlbumDataset(Dataset):
    def __init__(self, root: str, transform: A.Compose):
        self._ds        = datasets.ImageFolder(root)
        self._transform = transform
        self.classes    = self._ds.classes

    def __len__(self):
        return len(self._ds)

    def __getitem__(self, idx):
        path, label = self._ds.samples[idx]
        img    = cv2.imread(path)
        img    = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        tensor = self._transform(image=img)['image']
        return tensor, label


def _cpu_inference_ms(model_cpu: nn.Module) -> float:
    """วัด inference latency บน CPU (fair comparison ทุกโมเดล)"""
    model_cpu.eval()
    dummy = torch.zeros(1, 3, 224, 224)
    for _ in range(10):
        model_cpu(dummy)
    t0 = time.time()
    for _ in range(100):
        model_cpu(dummy)
    return (time.time() - t0) / 100 * 1000


def _resolve_val(root: Path) -> Path:
    for name in ('val', 'valid'):
        p = root / name
        if p.exists():
            return p
    raise FileNotFoundError(f'val/valid not found in {root}')


def _build_torch_model(backbone: str, num_classes: int) -> nn.Module:
    if backbone == 'shufflenet':
        net    = M.shufflenet_v2_x1_0(weights=M.ShuffleNet_V2_X1_0_Weights.DEFAULT)
        net.fc = nn.Linear(1024, num_classes)
        return net
    if backbone == 'mobilenet':
        net               = M.mobilenet_v3_small(weights=M.MobileNet_V3_Small_Weights.DEFAULT)
        net.classifier[3] = nn.Linear(1024, num_classes)
        return net
    raise ValueError(f'Unknown backbone: {backbone!r}')


def _train_torch_model(backbone: str) -> None:
    """Train loop สำหรับ shufflenet / mobilenet พร้อม early stopping"""
    _seed_everything(SEED)
    val_dir = _resolve_val(DATA_DIR)

    train_ds = AlbumDataset(str(DATA_DIR / 'train'), _transform)
    val_ds   = AlbumDataset(str(val_dir),            _transform)
    test_ds  = AlbumDataset(str(DATA_DIR / 'test'),  _transform)
    n_cls    = len(train_ds.classes)
    print(f'[{backbone}] {n_cls} classes | train={len(train_ds)} val={len(val_ds)} test={len(test_ds)}')
    print(f'classes: {train_ds.classes}')

    train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                              num_workers=4, pin_memory=True, persistent_workers=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False,
                              num_workers=4, pin_memory=True, persistent_workers=True)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False,
                              num_workers=4, pin_memory=True, persistent_workers=True)

    net       = _build_torch_model(backbone, n_cls).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(net.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=5)

    out_dir = OUT_DIR / backbone
    out_dir.mkdir(parents=True, exist_ok=True)

    best_val_acc  = 0.0
    no_improve    = 0
    stopped_epoch = MAX_EPOCHS

    for epoch in range(1, MAX_EPOCHS + 1):
        # ── train ──────────────────────────────────────────────────────────
        net.train()
        correct = total = 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            out  = net(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            correct += (out.detach().argmax(1) == y).sum().item()
            total   += len(y)
        train_acc = correct / total

        # ── val ────────────────────────────────────────────────────────────
        net.eval()
        with torch.no_grad():
            correct = total = 0
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                correct += (net(x).argmax(1) == y).sum().item()
                total   += len(y)
        val_acc = correct / total
        scheduler.step(val_acc)

        print(f'Epoch {epoch:3d}/{MAX_EPOCHS}  train={train_acc:.4f}  val={val_acc:.4f}  '
              f'no_improve={no_improve}/{PATIENCE}')

        if val_acc > best_val_acc:
            best_val_acc  = val_acc
            no_improve    = 0
            torch.save(net.state_dict(), out_dir / 'best.pt')
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            stopped_epoch = epoch
            print(f'[early stop] epoch {epoch} — val_acc ไม่ดีขึ้นใน {PATIENCE} epochs')
            break

    # ── test ───────────────────────────────────────────────────────────────
    net.load_state_dict(torch.load(out_dir / 'best.pt', map_location=device, weights_only=True))
    ms_cpu = _cpu_inference_ms(net.cpu())
    net    = net.to(device)
    net.eval()
    with torch.no_grad():
        all_preds, all_labels = [], []
        correct = total = 0
        for x, y in test_loader:
            x, y  = x.to(device), y.to(device)
            preds  = net(x).argmax(1)
            correct += (preds == y).sum().item()
            total   += len(y)
            all_preds  += preds.cpu().tolist()
            all_labels += y.cpu().tolist()
    test_acc = correct / total

    params_m = sum(p.numel() for p in net.parameters()) / 1e6
    size_mb  = (out_dir / 'best.pt').stat().st_size / 1e6

    print()
    print('='*55)
    print(f'[{backbone}] stopped={stopped_epoch}/{MAX_EPOCHS}  test={test_acc:.4f}  val={best_val_acc:.4f}')
    print(f'  {params_m:.2f}M params | {size_mb:.1f}MB | {ms_cpu:.2f}ms/img (CPU)')
    print()
    print(classification_report(all_labels, all_preds, target_names=train_ds.classes))
    print('Confusion Matrix:')
    print(confusion_matrix(all_labels, all_preds))

    save_summary(backbone, test_acc, best_val_acc, stopped_epoch, params_m, size_mb, ms_cpu)


device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {device}')

---
## Model 1 — YOLOv8n-cls
**Paradigm:** YOLO classification head (ต่อจาก backbone เดียวกับ YOLOv8 detector)  
**Input:** 224×224  
**Params:** ~1.45 M  
**Trainer:** Ultralytics (built-in early stopping via `patience=`)

In [ ]:
from ultralytics import YOLO
import pandas as pd

_dev    = '0' if torch.cuda.is_available() else 'cpu'
_YC_OUT = OUT_DIR / 'yolo_cls'
_YC_OUT.mkdir(parents=True, exist_ok=True)

# YOLO cls ต้องการ 'val/' — สร้าง symlink ถ้า dataset ใช้ชื่อ 'valid/'
val_dir  = _resolve_val(DATA_DIR)
val_link = DATA_DIR / 'val'
if not val_link.exists() and val_dir.name == 'valid':
    val_link.symlink_to(val_dir.resolve())

# ── Train ─────────────────────────────────────────────────────────────────
model = YOLO('yolov8n-cls.pt')
model.train(
    data     = str(DATA_DIR),
    epochs   = MAX_EPOCHS,
    imgsz    = 224,
    batch    = BATCH,
    seed     = SEED,
    device   = _dev,
    patience = PATIENCE,
    project  = str(_YC_OUT),
    name     = 'run',
    exist_ok = True,
    verbose  = False,
)

# ดึง path จริงจาก trainer (ป้องกัน path ผิดถ้า YOLO เปลี่ยน naming)
save_dir = Path(model.trainer.save_dir)
best_pt  = save_dir / 'weights' / 'best.pt'
print(f'Weights: {best_pt}')

# ── Accuracy ──────────────────────────────────────────────────────────────
test_met     = model.val(data=str(DATA_DIR), split='test', imgsz=224, device=_dev, verbose=False)
test_acc     = float(test_met.top1)

val_met      = model.val(data=str(DATA_DIR), split='val',  imgsz=224, device=_dev, verbose=False)
best_val_acc = float(val_met.top1)

# ── CPU inference speed (load fresh — ไม่ปนกับ model ที่ train แล้ว) ─────────
torch_model = YOLO(str(best_pt)).model.cpu().eval()
ms_cpu      = _cpu_inference_ms(torch_model)

# ── Model stats ───────────────────────────────────────────────────────────
params_m = sum(p.numel() for p in torch_model.parameters()) / 1e6
size_mb  = best_pt.stat().st_size / 1e6

# ── Stopped epoch ─────────────────────────────────────────────────────────
results_csv   = save_dir / 'results.csv'
stopped_epoch = len(pd.read_csv(results_csv)) if results_csv.exists() else MAX_EPOCHS

# ── Per-class report (load fresh อีกครั้ง — หลีกเลี่ยง tensor version error) ─
model_rep   = YOLO(str(best_pt))
test_dir    = DATA_DIR / 'test'
class_names = sorted([d.name for d in test_dir.iterdir() if d.is_dir()])
all_preds, all_labels = [], []
for idx, cls in enumerate(class_names):
    for img_p in (test_dir / cls).glob('*'):
        r = model_rep.predict(str(img_p), imgsz=224, verbose=False, device=_dev)
        all_preds.append(int(r[0].probs.top1))
        all_labels.append(idx)

print()
print('='*55)
print(f'[yolo_cls] stopped={stopped_epoch}/{MAX_EPOCHS}  test={test_acc:.4f}  val={best_val_acc:.4f}')
print(f'  {params_m:.2f}M params | {size_mb:.1f}MB | {ms_cpu:.2f}ms/img (CPU)')
print()
print(classification_report(all_labels, all_preds, target_names=class_names))
print('Confusion Matrix:')
print(confusion_matrix(all_labels, all_preds))

save_summary('yolo_cls', test_acc, best_val_acc, stopped_epoch, params_m, size_mb, ms_cpu)

---
## Model 2 — ShuffleNet V2 x1.0
**Paradigm:** Lightweight CNN (channel shuffle สำหรับ efficient inference)  
**Input:** 224×224  
**Params:** ~1.27 M  
**Trainer:** PyTorch custom loop พร้อม early stopping + ReduceLROnPlateau

In [ ]:
_train_torch_model('shufflenet')

---
## Model 3 — MobileNet V3 Small
**Paradigm:** Depthwise separable convolution + SE attention  
**Input:** 224×224  
**Params:** ~1.53 M  
**Trainer:** PyTorch custom loop พร้อม early stopping + ReduceLROnPlateau

In [ ]:
_train_torch_model('mobilenet')

---
## Classification Model Comparison

เปรียบเทียบ 3 backbone บน dataset เดียวกัน — split/seed/epoch/patience เหมือนกัน

| | YOLOv8n-cls | ShuffleNet V2 | MobileNet V3 |
|---|---|---|---|
| Architecture | YOLO head | Channel shuffle | Depthwise + SE |
| Pretrained | COCO | ImageNet | ImageNet |
| Trainer | Ultralytics | PyTorch | PyTorch |

In [ ]:
import pandas as pd

df = pd.read_csv(_CSV)
df = df.rename(columns={
    'backbone':         'Backbone',
    'test_acc':         'Test Acc',
    'best_val_acc':     'Best Val Acc',
    'stopped_epoch':    'Stopped Ep.',
    'max_epochs':       'Max Ep.',
    'params_M':         'Params (M)',
    'size_mb':          'Size (MB)',
    'ms_per_image_cpu': 'ms/img (CPU)',
})

print(df.to_string(index=False))
print(f'\n[saved] {_CSV}')